In [1]:
import os
import geopandas as gpd
from shapely.geometry import box
from zensvi.download import MLYDownloader
from paddleocr import PaddleOCR
import pandas as pd

In [2]:
def acquire_street_view_data():
    """
    Downloads street-level imagery for sociolinguistic OCR analysis.
    Leverages Mapillary's open-source database to ensure ethical data sourcing
    and compliance with GIS research standards.
    """
    
    # 1. Initialize the Mapillary Downloader
    # Replace with your actual Mapillary Client Token
    mly_api_key = "MLY|26710645935302388|1496476136fe59863a3f5ceda4c38b71" 
    

In [3]:
def acquire_street_view_data():
    # 1. Authenticate with Mapillary
    mly_api_key = "MLY|26710645935302388|1496476136fe59863a3f5ceda4c38b71" 
    
    try:
        downloader = MLYDownloader(mly_api_key=mly_api_key)
        print("Successfully authenticated with Mapillary API.")
    except Exception as e:
        print(f"Authentication failed. Please check your API key. Error: {e}")
        return

    # 2. Define the Target Geographies (Bounding Boxes)
    seoul_neighborhoods = {
        "Gangnam": [127.0210, 37.4950, 127.0360, 37.5050],
        "Eunpyeong": [126.9150, 37.6100, 126.9350, 37.6250],
        "Hongdae": [126.9160, 37.5480, 126.9290, 37.5590],
        "Itaewon": [126.9850, 37.5310, 127.0000, 37.5410],
        "Jongno": [126.9810, 37.5710, 126.9910, 37.5810],
        "Seongsu": [127.0490, 37.5390, 127.0600, 37.5490]
    }

    # 3. Execute the Batch Download using Shapefiles
    # .items() unpacks the dictionary into the name and the coordinate list
    for place_name, coords in seoul_neighborhoods.items():
        save_folder = f"./data/raw_images/{place_name.lower()}"
        print(f"\n--- Initiating data pull for: {place_name} ---")
        
        # Ensure the directory exists
        os.makedirs(save_folder, exist_ok=True)
        
        # -- The Bounding Box Conversion --
        # Unpack the coordinates into Shapely format
        min_lon, min_lat, max_lon, max_lat = coords
        bbox_geom = box(min_lon, min_lat, max_lon, max_lat)
        
        # Create a GeoDataFrame and save it as a temporary shapefile in the target folder
        gdf = gpd.GeoDataFrame({'geometry': [bbox_geom]}, crs="EPSG:4326")
        temp_shp_path = os.path.join(save_folder, "target_boundary.shp")
        gdf.to_file(temp_shp_path)
        
        # Download using the dynamically generated shapefile
        try:
            downloader.download_svi(
                save_folder, 
                input_shp_file=temp_shp_path, # Feed the shapefile to ZenSVI
                buffer=0 
            )
            print(f"Data acquisition complete. Images saved to: {save_folder}")
            
        except Exception as e:
            print(f"Failed to retrieve data for {place_name}. Error: {e}")

if __name__ == "__main__":
    acquire_street_view_data()

Successfully authenticated with Mapillary API.

--- Initiating data pull for: Gangnam ---
Getting pids...
update_pids is set to False. So the following csv file will be used: data\raw_images\gangnam\mly_pids.csv
The panorama URLs have been read from the cache


The cache directory has been deleted
Data acquisition complete. Images saved to: ./data/raw_images/gangnam

--- Initiating data pull for: Eunpyeong ---
Getting pids...


Loading cache files: 0it [00:00, ?it/s]


[Vector Tiles API] Fetching 2 tiles for images ...


Processing tiles: 100%|██████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  8.76it/s]
Filtering data: 0it [00:00, ?it/s]


There is no panorama ID to download
There is no panorama ID to download within the given input parameters
The cache directory has been deleted
Data acquisition complete. Images saved to: ./data/raw_images/eunpyeong

--- Initiating data pull for: Hongdae ---
Getting pids...
update_pids is set to False. So the following csv file will be used: data\raw_images\hongdae\mly_pids.csv
The panorama URLs have been read from the cache


The cache directory has been deleted
Data acquisition complete. Images saved to: ./data/raw_images/hongdae

--- Initiating data pull for: Itaewon ---
Getting pids...
update_pids is set to False. So the following csv file will be used: data\raw_images\itaewon\mly_pids.csv
The panorama URLs have been read from the cache


The cache directory has been deleted
Data acquisition complete. Images saved to: ./data/raw_images/itaewon

--- Initiating data pull for: Jongno ---
Getting pids...
update_pids is set to False. So the following csv file will be used: data\raw_images\jongno\mly_pids.csv
The panorama URLs have been read from the cache


The cache directory has been deleted
Data acquisition complete. Images saved to: ./data/raw_images/jongno

--- Initiating data pull for: Seongsu ---
Getting pids...
update_pids is set to False. So the following csv file will be used: data\raw_images\seongsu\mly_pids.csv
The panorama URLs have been read from the cache


The cache directory has been deleted
Data acquisition complete. Images saved to: ./data/raw_images/seongsu


In [8]:
def extract_street_sign_text(base_dir="./data/raw_images", sample_limit=5):
    print("Initializing OCR Engine (with oneDNN disabled for Windows stability)...")
    # Added enable_mkldnn=False to bypass the C++ oneDNN crash
    ocr = PaddleOCR(use_angle_cls=True, lang='korean', enable_mkldnn=False)
    
    extracted_data = []

    neighborhoods = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

    for neighborhood in neighborhoods:
        print(f"\nScanning {neighborhood.title()}...")
        neighborhood_path = os.path.join(base_dir, neighborhood)
        
        image_paths = []
        for root, dirs, files in os.walk(neighborhood_path):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                    image_paths.append(os.path.join(root, file))
        
        print(f"-> Found {len(image_paths)} image files.")
        
        test_images = image_paths[:sample_limit]
        
        for img_path in test_images:
            try:
                result = ocr.ocr(img_path)
                
                if result and result[0]:
                    for line in result[0]:
                        detected_text = line[1][0]
                        confidence = line[1][1]
                        
                        if confidence > 0.60:
                            extracted_data.append({
                                "neighborhood": neighborhood,
                                "image_file": os.path.basename(img_path),
                                "detected_text": detected_text,
                                "confidence_score": round(confidence, 4)
                            })
            except Exception as e:
                print(f"Error processing {os.path.basename(img_path)}: {e}")

    if extracted_data:
        df = pd.DataFrame(extracted_data)
        output_csv = "./data/seoul_typography_sample.csv"
        df.to_csv(output_csv, index=False, encoding='utf-8-sig') 
        print(f"\nSuccess! Extracted {len(df)} lines of text.")
        print(f"Data saved to: {output_csv}")
        return df
    else:
        print("\nNo text detected with high confidence in the sample batch.")
        return None

if __name__ == "__main__":
    df_results = extract_street_sign_text()
    if df_results is not None:
        display(df_results.head(10))

C:\Users\barre\AppData\Local\Temp\ipykernel_38132\4211435007.py:4: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  ocr = PaddleOCR(use_angle_cls=True, lang='korean', enable_mkldnn=False)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\barre\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\barre\.paddlex\official_models\UVDoc`.


Initializing OCR Engine (with oneDNN disabled for Windows stability)...


Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\barre\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\barre\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('korean_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\barre\.paddlex\official_models\korean_PP-OCRv5_mobile_rec`.



Scanning Eunpyeong...
-> Found 0 image files.

Scanning Gangnam...
-> Found 7983 image files.


C:\Users\barre\AppData\Local\Temp\ipykernel_38132\4211435007.py:26: DeprecationWarning: Please use `predict` instead.
  result = ocr.ocr(img_path)


Error processing 1000376121507852.png: string index out of range
Error processing 1000382532027699.png: string index out of range
Error processing 1001808401302124.png: string index out of range
Error processing 1002877101819817.png: string index out of range
Error processing 1003409398400267.png: string index out of range

Scanning Hongdae...
-> Found 383 image files.
Error processing 1001417597060412.png: string index out of range
Error processing 1009066186531740.png: string index out of range
Error processing 1009889163084856.png: string index out of range
Error processing 1015045829028363.png: string index out of range
Error processing 1024345183013103.png: string index out of range

Scanning Itaewon...
-> Found 1147 image files.
Error processing 1001515058046727.png: string index out of range
Error processing 1001588124827923.png: string index out of range
Error processing 1002983248211429.png: string index out of range
Error processing 10038031962891589.png: string index out of 

In [11]:
# Copy and run this entire block in your Jupyter cell
gitignore_content = """
# Ignore virtual environments
my_zensvi/

# Ignore raw image datasets and heavy shapefiles
data/raw_images/
data/balanced_images/
*.shp
*.shx
*.dbf
*.prj

# Ignore Jupyter temp files
.ipynb_checkpoints/
__pycache__/
"""

# This code tells Python to write that text into a file named '.gitignore'
with open(".gitignore", "w") as f:
    f.write(gitignore_content.strip())

print("Successfully created .gitignore file!")

Successfully created .gitignore file!
